# SerpentGuard AI — Extended Sources Data Pull

Pulls snake photos and masks from sources beyond our iNaturalist build and saves them, with manifests, as the private dataset `<your username>/serpentguard-extended-sources`. `09_combine_and_reclean.ipynb` then drops anything we already have and adds the rest to training only.

| Source | What we take | Status |
|---|---|---|
| SnakeCLEF 2023, iNaturalist part (`munhamrehman/snakeclef2023-500px`) | photos of our 208 species whose observation is not already in our dataset | runs |
| `aishanikaggle/indian-snake` (CC BY-NC-SA) | the six folders whose common name points to one species | runs |
| `adityasharma01/snake-dataset-india` | nothing: labelled only venomous or non-venomous | skipped |
| Open Images V7 | human-drawn snake masks and boxes from the validation and test splits | runs |
| GBIF India | needs a GBIF account (`GBIF_USER`, `GBIF_PWD`) | only with credentials |
| Roboflow Universe | needs `ROBOFLOW_API_KEY` | only with a key |

Internet must be on. Extra photos keep their own licences, so this dataset stays private.

In [1]:
import numpy as np, pandas as pd, requests, json, os, shutil, io, zipfile, logging
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
os.environ["TQDM_DISABLE"] = "1"  # hundreds of single-file downloads would flood the output with progress bars
import kagglehub
from PIL import Image
logging.getLogger("kagglehub").setLevel(logging.ERROR)

SPLITS_HANDLE = "puneetbajajiitm/serpentguard-m2-splits"
KAGGLE_USERNAME = kagglehub.whoami()["username"]
OUT = Path("/kaggle/working/serpentguard-extended-sources")
(OUT / "images").mkdir(parents=True, exist_ok=True)
UPLOAD = True

# Scheduled Kaggle runs can only read attached datasets: attach the splits, SnakeCLEF and India datasets
splits_root = Path(kagglehub.dataset_download(SPLITS_HANDLE))
classes_df = pd.read_csv(splits_root / "classes.csv")
ours_df = pd.read_csv(splits_root / "splits.csv", usecols=["obs_id", "photo_id"], dtype=str)
OUR_SPECIES = set(classes_df["species"])
OUR_OBS, OUR_PHOTOS = set(ours_df["obs_id"]), set(ours_df["photo_id"])
manifest_rows = []
print(f"Our dataset: {len(OUR_SPECIES)} classes, {len(ours_df):,} photos from {len(OUR_OBS):,} observations")

Our dataset: 208 classes, 181,503 photos from 95,877 observations


## Source 1: SnakeCLEF 2023 (iNaturalist part, 500 px)
`munhamrehman/snakeclef2023-500px` holds only the iNaturalist part of SnakeCLEF 2023. Its `observation_id` is the iNaturalist observation id and its file name is the photo id, so photos we already have are dropped by id first. The 18.6 GB dataset is attached (Kaggle mounts it rather than copying it), and only the files of new observations are copied.

In [2]:
SNAKECLEF_HANDLE = "munhamrehman/snakeclef2023-500px"
SNAKECLEF_META = "SnakeCLEF2023-TrainMetadata-iNat.csv"
SNAKECLEF_IMAGES = "SnakeCLEF2023-train-medium_size/SnakeCLEF2023-medium_size"

sclef_root = Path(kagglehub.dataset_download(SNAKECLEF_HANDLE))  # resolve once: one request per file hits Kaggle's rate limit
sclef_df = pd.read_csv(sclef_root / SNAKECLEF_META)
sclef_df["photo_id"] = sclef_df["image_path"].str.extract(r"/(\d+)\.\w+$")[0]
ours = sclef_df[sclef_df["binomial_name"].isin(OUR_SPECIES)]
new = ours[~ours["observation_id"].astype(str).isin(OUR_OBS) & ~ours["photo_id"].isin(OUR_PHOTOS)]
print(f"SnakeCLEF 2023 iNaturalist rows: {len(sclef_df):,}; our species: {len(ours):,} ({(ours['code'] == 'IN').sum():,} from India)")
print(f"Already in our dataset: {len(ours) - len(new):,}; new: {len(new):,} ({(new['code'] == 'IN').sum()} from India)")

def fetch_snakeclef(row):
    dest = OUT / "images" / "snakeclef2023" / f"{row.photo_id}{Path(row.image_path).suffix.lower()}"
    try:
        shutil.copy(sclef_root / SNAKECLEF_IMAGES / row.image_path, dest)
    except Exception as err:
        return None, f"{type(err).__name__}"
    return dest, None

(OUT / "images" / "snakeclef2023").mkdir(parents=True, exist_ok=True)
with ThreadPoolExecutor(8) as pool:
    fetched = list(pool.map(fetch_snakeclef, new.itertuples()))
failed = 0
for row, (dest, error) in zip(new.itertuples(), fetched):
    if dest is None:
        failed += 1
        continue
    manifest_rows.append({"source": "snakeclef2023", "species": row.binomial_name, "file": str(dest.relative_to(OUT)),
                          "obs_id": str(row.observation_id), "photo_id": row.photo_id, "country": row.code,
                          "license": "", "attribution": "",
                          "url": f"https://www.inaturalist.org/observations/{row.observation_id}"})
print(f"SnakeCLEF photos saved: {len(new) - failed:,}; failed downloads: {failed}")

SnakeCLEF 2023 iNaturalist rows: 154,301; our species: 16,171 (4,619 from India)
Already in our dataset: 15,461; new: 710 (62 from India)
SnakeCLEF photos saved: 704; failed downloads: 6


## Source 2: Kaggle India-specific datasets
`aishanikaggle/indian-snake` (CC BY-NC-SA) names its folders by common name. Only six names point to a single species in India. Folders such as "cobra", "pit viper", "python", "keelback", "kukri", "racer", "wolf snake", "Indian boa" and "Indian cat" cover several of our species, so labelling each as one species would add wrong labels; they are left out, as are the `train/` and `valid/` folders, which have no species. `adityasharma01/snake-dataset-india` is labelled only venomous or non-venomous, so it adds no species labels and is not used.

In [3]:
INDIA_HANDLE = "aishanikaggle/indian-snake"
INDIA_LABEL_MAP = {
    "COMMON_KRAIT-VENOMOUS": "Bungarus caeruleus",
    "RUSSELL_VIPPER_VENOMOUS": "Daboia russelii",
    "SAW_SCALED_VIPPER_VENOMOUS": "Echis carinatus",
    "RAT_SNAKE_NON-VENOMOUS": "Ptyas mucosa",
    "COMMON_TRINKET_NON-VENOUMOUS": "Coelognathus helena",
    "BLACK_HEADED_ROYAL_SNAKE_NON-VENOUMOUS": "Spalerosophis atriceps",
}
IMAGE_SUFFIXES = (".jpg", ".jpeg", ".png")

india_root = Path(kagglehub.dataset_download(INDIA_HANDLE)) / "snake_dataset"
india_count = 0
for folder in sorted(p for p in india_root.iterdir() if p.is_dir()):
    images = sorted(p for p in folder.rglob("*") if p.suffix.lower() in IMAGE_SUFFIXES)
    species = INDIA_LABEL_MAP.get(folder.name)
    if species is None or species not in OUR_SPECIES:
        print(f"  skip {folder.name}: {len(images)} images ({'no single species' if species is None else 'not one of our classes'})")
        continue
    dest_dir = OUT / "images" / "indian_snake" / folder.name
    dest_dir.mkdir(parents=True, exist_ok=True)
    for k, img in enumerate(images):
        dest = dest_dir / f"{k:04d}{img.suffix.lower()}"
        shutil.copy(img, dest)
        manifest_rows.append({"source": "indian_snake", "species": species, "file": str(dest.relative_to(OUT)),
                              "obs_id": "", "photo_id": "", "country": "IN", "license": "cc-by-nc-sa",
                              "attribution": "aishanikaggle/indian-snake", "url": f"https://www.kaggle.com/datasets/{INDIA_HANDLE}"})
    india_count += len(images)
    print(f"  {folder.name} -> {species}: {len(images)} images")
print(f"India dataset photos saved: {india_count:,}")

  BLACK_HEADED_ROYAL_SNAKE_NON-VENOUMOUS -> Spalerosophis atriceps: 102 images
  skip COBRA_VENOUMOUS: 125 images (no single species)
  COMMON_KRAIT-VENOMOUS -> Bungarus caeruleus: 106 images
  COMMON_TRINKET_NON-VENOUMOUS -> Coelognathus helena: 95 images
  skip INDIAN_BOA_NON-VENOUMOUS: 120 images (no single species)
  skip INDIAN_CAT_VEN: 121 images (no single species)
  skip KEELBACK_NON-VENOMOUS: 95 images (no single species)
  skip KUKRI_NON-VENOMOUS: 109 images (no single species)
  skip PIT_VIPPER_VENOMOUS: 152 images (no single species)
  skip PYTHON_NON-VENOMOUS: 319 images (no single species)
  skip RACER_SNAKE_NON-VENOMOUS: 445 images (no single species)
  RAT_SNAKE_NON-VENOMOUS -> Ptyas mucosa: 435 images
  RUSSELL_VIPPER_VENOMOUS -> Daboia russelii: 353 images
  SAW_SCALED_VIPPER_VENOMOUS -> Echis carinatus: 243 images
  skip WOLF_SNAKE_NON-VENOMOUS: 107 images (no single species)
  skip train: 777 images (no single species)
  skip valid: 192 images (no single species)
In

## Source 3: Open Images V7 — human-drawn snake masks and boxes
Open Images has human-verified segmentation masks for the class Snake (`/m/078jl`) in its validation and test splits. They carry no species, so they are not classifier training photos. They are the ground truth we lacked for scoring segmentation: our SAM masks are automatic pseudo-labels on training photos only. Photos come from the public Open Images bucket, with each photo's Flickr licence and author.

In [4]:
OI_BASE = "https://storage.googleapis.com/openimages"
OI_IMAGES = "https://open-images-dataset.s3.amazonaws.com"
OI_SNAKE = "/m/078jl"
OI_IMAGE_LISTS = {"validation": "2018_04/validation/validation-images-with-rotation.csv",
                  "test": "2018_04/test/test-images-with-rotation.csv"}
OI_DIR = OUT / "open_images"
oi_rows, oi_boxes = [], []

def get_csv(url):
    response = requests.get(url, timeout=300)
    response.raise_for_status()
    return pd.read_csv(io.BytesIO(response.content))

for split, image_list in OI_IMAGE_LISTS.items():
    seg = get_csv(f"{OI_BASE}/v5/{split}-annotations-object-segmentation.csv")
    seg = seg[seg["LabelName"] == OI_SNAKE]
    boxes = get_csv(f"{OI_BASE}/v5/{split}-annotations-bbox.csv")
    oi_boxes.append(boxes[(boxes["LabelName"] == OI_SNAKE) & boxes["ImageID"].isin(seg["ImageID"])].assign(split=split))
    info = get_csv(f"{OI_BASE}/{image_list}").set_index("ImageID")
    (OI_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (OI_DIR / split / "masks").mkdir(parents=True, exist_ok=True)

    wanted = set(seg["MaskPath"])
    for shard in "0123456789abcdef":
        response = requests.get(f"{OI_BASE}/v5/{split}-masks/{split}-masks-{shard}.zip", timeout=600)
        response.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(response.content)) as masks_zip:
            for name in wanted & set(masks_zip.namelist()):
                (OI_DIR / split / "masks" / name).write_bytes(masks_zip.read(name))

    def fetch_image(image_id):
        dest = OI_DIR / split / "images" / f"{image_id}.jpg"
        response = requests.get(f"{OI_IMAGES}/{split}/{image_id}.jpg", timeout=120)
        if response.status_code != 200:
            return None
        dest.write_bytes(response.content)
        return dest

    with ThreadPoolExecutor(8) as pool:
        images = dict(zip(seg["ImageID"].unique(), pool.map(fetch_image, seg["ImageID"].unique())))
    for row in seg.itertuples():
        image_path, mask_path = images.get(row.ImageID), OI_DIR / split / "masks" / row.MaskPath
        if image_path is None or not mask_path.exists():
            continue
        with Image.open(image_path) as im, Image.open(mask_path) as mk:
            image_size, mask_size = im.size, mk.size
        meta = info.loc[row.ImageID]
        oi_rows.append({"split": split, "image_id": row.ImageID, "image_file": str(image_path.relative_to(OUT)),
                        "mask_file": str(mask_path.relative_to(OUT)), "box_xmin": row.BoxXMin, "box_xmax": row.BoxXMax,
                        "box_ymin": row.BoxYMin, "box_ymax": row.BoxYMax, "image_width": image_size[0],
                        "image_height": image_size[1], "mask_width": mask_size[0], "mask_height": mask_size[1],
                        "rotation": meta["Rotation"], "license": meta["License"], "author": meta["Author"],
                        "url": meta["OriginalLandingURL"]})
    print(f"Open Images {split}: {len(seg)} snake masks on {seg['ImageID'].nunique()} photos; "
          f"saved {sum(r['split'] == split for r in oi_rows)} masks")

Open Images validation: 63 snake masks on 58 photos; saved 63 masks
Open Images test: 233 snake masks on 226 photos; saved 233 masks


## Source 4: GBIF India snake occurrences (with images)

In [5]:
# GBIF bulk download for India snake occurrences with images
# Requires GBIF account credentials

GBIF_USER = os.environ.get("GBIF_USER", "")
GBIF_PWD  = os.environ.get("GBIF_PWD", "")

if GBIF_USER and GBIF_PWD:
    import base64
    creds = base64.b64encode(f"{GBIF_USER}:{GBIF_PWD}".encode()).decode()
    
    # Request GBIF download (async - takes minutes to prepare)
    predicate = {
        "type": "and",
        "predicates": [
            {"type": "equals", "key": "ORDER", "value": "Squamata"},
            {"type": "equals", "key": "CLASS", "value": "Reptilia"},
            {"type": "equals", "key": "COUNTRY", "value": "IN"},
            {"type": "equals", "key": "HAS_COORDINATE", "value": "true"},
            {"type": "equals", "key": "HAS_GEOSPATIAL_ISSUE", "value": "false"},
            {"type": "equals", "key": "MEDIA_TYPE", "value": "StillImage"},
        ]
    }
    
    resp = requests.post(
        "https://api.gbif.org/v1/occurrence/download/request",
        json={"predicate": predicate},
        headers={"Authorization": f"Basic {creds}", "Content-Type": "application/json"}
    )
    
    if resp.status_code == 201:
        download_key = resp.text.strip()
        print(f"GBIF download requested. Key: {download_key}")
        print("Poll: https://api.gbif.org/v1/occurrence/download/{download_key}")
        print("Download will be ready in 5-30 minutes via email link.")
        
        # Save key for later
        with open("/kaggle/working/gbif_download_key.txt", "w") as f:
            f.write(download_key)
    else:
        print(f"GBIF request failed: {resp.status_code} {resp.text[:200]}")
else:
    print("GBIF credentials not set. Set env vars GBIF_USER and GBIF_PWD.")
    print("Register at: https://www.gbif.org/user/profile")
    print("GBIF India snake occurrence data also available at:")
    print("  https://www.gbif.org/occurrence/search?country=IN&taxon_key=11418114&media_type=StillImage")

GBIF credentials not set. Set env vars GBIF_USER and GBIF_PWD.
Register at: https://www.gbif.org/user/profile
GBIF India snake occurrence data also available at:
  https://www.gbif.org/occurrence/search?country=IN&taxon_key=11418114&media_type=StillImage


## Source 5: Roboflow Universe snake detection

In [6]:
# Roboflow Universe snake detection datasets
# Known relevant projects:
ROBOFLOW_PROJECTS = [
    ("snakes-zjtwh", "snakes-zjtwh"),           # snake detection
    ("snake-detection-dataset", "ygzaw"),        # general snake detection
    ("serpent-detection", "serpent-detection"),  # cobra/viper focus
]

ROBOFLOW_KEY = os.environ.get("ROBOFLOW_API_KEY", "")
rf_rows = []

if ROBOFLOW_KEY:
    try:
        from roboflow import Roboflow
        rf = Roboflow(api_key=ROBOFLOW_KEY)
        
        output_det = Path("/kaggle/working/roboflow_detection")
        output_det.mkdir(exist_ok=True)
        
        for workspace, project_name in ROBOFLOW_PROJECTS:
            try:
                project = rf.workspace(workspace).project(project_name)
                version = project.versions()[0]
                dataset = version.download("yolov8", location=str(output_det / project_name))
                print(f"Downloaded: {workspace}/{project_name}")
                
                # Count images and boxes
                imgs = list((output_det / project_name).glob("**/*.jpg"))
                labels = list((output_det / project_name).glob("**/*.txt"))
                rf_rows.append({"project": project_name, "images": len(imgs), "labels": len(labels)})
            except Exception as e:
                print(f"  Failed {workspace}/{project_name}: {e}")
    except ImportError:
        print("roboflow not installed. Install: pip install roboflow")
else:
    print("ROBOFLOW_API_KEY not set. Visit: https://roboflow.com/universe")
    print("Search 'snake detection' to find relevant datasets.")
    
    # Fallback: list known public datasets
    public_datasets = [
        "https://universe.roboflow.com/lj-gkmpv/snakes-zjtwh",
        "https://universe.roboflow.com/summerflight/common-snake-species-dataset-for-object-detection",
    ]
    print("Known public datasets:")
    for url in public_datasets:
        print(f"  {url}")

rf_df = pd.DataFrame(rf_rows) if rf_rows else pd.DataFrame()
print(f"Roboflow detection datasets: {len(rf_df)}")

ROBOFLOW_API_KEY not set. Visit: https://roboflow.com/universe
Search 'snake detection' to find relevant datasets.
Known public datasets:
  https://universe.roboflow.com/lj-gkmpv/snakes-zjtwh
  https://universe.roboflow.com/summerflight/common-snake-species-dataset-for-object-detection
Roboflow detection datasets: 0


## Summary: save the extended sources dataset
`extended_manifest.csv` lists the species-labelled photos (SnakeCLEF and the India dataset) for notebook 09. `open_images_snake_masks.csv` lists the Open Images masks with their photo, box, licence and author; `open_images_snake_boxes.csv` has every snake box on those photos.

In [7]:
extended_df = pd.DataFrame(manifest_rows)
extended_df.to_csv(OUT / "extended_manifest.csv", index=False)
oi_df = pd.DataFrame(oi_rows)
oi_df.to_csv(OUT / "open_images_snake_masks.csv", index=False)
pd.concat(oi_boxes).to_csv(OUT / "open_images_snake_boxes.csv", index=False)

print(f"Species-labelled photos: {len(extended_df):,}")
print(extended_df.groupby(["source", "species"]).size().to_string())
same_shape = ((oi_df["image_width"] * oi_df["mask_height"] - oi_df["image_height"] * oi_df["mask_width"]).abs()
              <= 0.02 * oi_df["image_width"] * oi_df["mask_height"])
print(f"Open Images masks: {len(oi_df)} on {oi_df['image_id'].nunique()} photos; "
      f"{same_shape.sum()} have the photo's aspect ratio; rotation tags: {oi_df['rotation'].fillna(0).value_counts().to_dict()}")
print("Open Images licences:", oi_df["license"].value_counts().to_dict())

(OUT / "README.md").write_text("""# SerpentGuard extended sources (private, course use only)

Made by notebooks/08_extended_sources.ipynb.

- extended_manifest.csv: species-labelled photos from SnakeCLEF 2023 (iNaturalist part, only
  observations not already in serpentguard-indian-snakes) and aishanikaggle/indian-snake (six
  single-species folders). Files are under images/. Use notebook 09 before training on them.
- open_images_snake_masks.csv: human-drawn Open Images V7 snake masks (validation and test splits)
  with photo, box, licence and author. For scoring segmentation, not for species training.
- open_images_snake_boxes.csv: every Open Images snake box on those photos.
""")
if UPLOAD:
    kagglehub.dataset_upload(f"{KAGGLE_USERNAME}/serpentguard-extended-sources", str(OUT), version_notes="08_extended_sources")

print("\nNext steps:")
print("1. Run 09_combine_and_reclean.ipynb to drop duplicates and add the new photos to training")
print("2. Use open_images_snake_masks.csv to score the SAM masks and the model's mask head")

Species-labelled photos: 2,038
source         species                     
indian_snake   Bungarus caeruleus              106
               Coelognathus helena              95
               Daboia russelii                 353
               Echis carinatus                 243
               Ptyas mucosa                    435
               Spalerosophis atriceps          102
snakeclef2023  Acrochordus granulatus            1
               Ahaetulla farnsworthi             2
               Ahaetulla isabellina              1
               Ahaetulla prasina                 8
               Ahaetulla sahyadrensis            1
               Boiga beddomei                    3
               Boiga cyanea                      4
               Boiga forsteni                    1
               Boiga multifasciata               1
               Boiga multomaculata               1
               Boiga nuchalis                    1
               Boiga siamensis                   4
       